# Creating the training/validation/test dataset USING ONLY **ROI IMAGES** AND **LABELS**

## CBIS-DDSM ROI Preprocessing and Patient-Level Splitting

This notebook prepares the CBIS-DDSM (Curated Breast Imaging Subset of DDSM) dataset for ROI classification (Calcification vs. Mass).

We will:

1. Load ROI images for calcification and mass cases.

2. Apply preprocessing (contrast enhancement, resizing, padding).

3. Perform patient-level separation to ensure no data leakage.

4. Combine all subsets for CNN training.

### Step 1 — Imports and Setup

In [1]:
import os
import cv2
import glob
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

### Step 2 — Define the Preprocessor Class

The class below:

- Loads and enhances grayscale ROI images.

- Resizes them to a fixed shape with padding to preserve aspect ratio.

- Splits the dataset at the **patient level**, ensuring that ROIs from the same patient are never shared between training and validation sets.

In [2]:
class CBIS_ROI_ClassifierPreprocessor:
    def __init__(self, img_size=(224, 224)):
        """
        Preprocessor for CBIS-DDSM ROI classification (calc vs mass).

        Args:
            img_size (tuple): Target image size (height, width).
        """
        self.img_size = img_size
        self.class_mapping = {'calc': 0, 'mass': 1}  # Binary labels

    def load_image(self, image_path):
        """Load a single image in grayscale."""
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Could not load image: {image_path}")
        return img

    def enhance_contrast(self, img):
        """Enhance contrast using CLAHE — good for mammogram ROIs."""
        if img.dtype != np.uint8:
            img = (img * 255).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return clahe.apply(img)

    def resize_with_padding(self, img):
        """
        Resize image while preserving aspect ratio.
        Pads the image to match the desired shape.
        """
        h, w = img.shape[:2]
        target_h, target_w = self.img_size
        scale = min(target_w / w, target_h / h)
        new_w, new_h = int(w * scale), int(h * scale)
        resized_img = cv2.resize(img, (new_w, new_h))
        padded_img = np.zeros((target_h, target_w), dtype=resized_img.dtype)
        x_offset = (target_w - new_w) // 2
        y_offset = (target_h - new_h) // 2
        padded_img[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized_img
        return padded_img

    def process_image(self, image_path):
        """Apply all preprocessing steps to one image."""
        img = self.load_image(image_path)
        img = self.enhance_contrast(img)
        img = self.resize_with_padding(img)
        return img




    def load_dataset_patient_level(self, train_folder, test_folder, label_name, train_ratio=0.8, seed=42):
        """
        Load dataset with patient-level separation.

        Args:
            train_folder (str): Directory containing training patient subfolders.
            test_folder (str): Directory containing test patient subfolders.
            label_name (str): 'calc' or 'mass'.
            train_ratio (float): Proportion of training patients (rest go to validation).
            seed (int): Random seed for reproducibility.

        Returns:
            X_train, y_train, X_val, y_val, X_test, y_test
        """
        # --- TRAIN / VAL ---
        patient_ids = [d for d in os.listdir(train_folder) if os.path.isdir(os.path.join(train_folder, d))]
        train_patients, val_patients = train_test_split(
            patient_ids, test_size=1-train_ratio, random_state=seed
        )

        def load_patients(folder, patients):
            X, y = [], []
            for pid in tqdm(patients, desc=f"Processing {label_name} patients in {folder}"):
                patient_folder = os.path.join(folder, pid)
                image_paths = glob.glob(os.path.join(patient_folder, "*.png"))
                for image_path in image_paths:
                    try:
                        img = self.process_image(image_path)
                        X.append(img)
                        y.append(self.class_mapping[label_name])
                    except Exception as e:
                        print(f"Error processing {image_path}: {e}")
            X = np.expand_dims(np.array(X), axis=-1)
            y = np.array(y)
            return X, y

        X_train, y_train = load_patients(train_folder, train_patients)
        X_val, y_val = load_patients(train_folder, val_patients)

        # --- TEST ---
        test_patient_ids = [d for d in os.listdir(test_folder) if os.path.isdir(os.path.join(test_folder, d))]
        X_test, y_test = load_patients(test_folder, test_patient_ids)

        print(f"{label_name} -> Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        return X_train, y_train, X_val, y_val, X_test, y_test


### Step 3 — Load Dataset with Patient-Level Splitting (upwards

We’ll now define a method that:

- Loads all patient folders.

- Splits them into **train / validation / test** sets based on patient IDs.

- Loads all ROI .png files for each patient.

- Applies preprocessing.

### Step 4 — Define Dataset Paths


Now we specify the paths to each subset.
Each folder should contain patient subfolders (e.g. P_1234/roi_1.png, P_1234/roi_2.png, etc.)

In [3]:
base_dir = r"D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks"
preprocessor = CBIS_ROI_ClassifierPreprocessor(img_size=(224, 224))

# Directories for each class
calc_train_dir = os.path.join(base_dir, "calc_case_description_train_set_png", "roi_crops")
calc_test_dir  = os.path.join(base_dir, "calc_case_description_test_set_png", "roi_crops")
mass_train_dir = os.path.join(base_dir, "mass_case_description_train_set_png", "roi_crops")
mass_test_dir  = os.path.join(base_dir, "mass_case_description_test_set_png", "roi_crops")


### Step 5 — Load and Preprocess Data
We now load both calcification and mass datasets, applying the same preprocessing pipeline.

In [11]:
X_train_calc, y_train_calc, X_val_calc, y_val_calc, X_test_calc, y_test_calc = preprocessor.load_dataset_patient_level(
    calc_train_dir, calc_test_dir, "calc", train_ratio=0.78
)
X_train_mass, y_train_mass, X_val_mass, y_val_mass, X_test_mass, y_test_mass = preprocessor.load_dataset_patient_level(
    mass_train_dir, mass_test_dir, "mass", train_ratio=0.78
)


Processing calc patients in D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks\calc_case_description_t
Processing calc patients in D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks\calc_case_description_t
Processing calc patients in D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks\calc_case_description_t


calc -> Train: 1214, Val: 332, Test: 326


Processing mass patients in D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks\mass_case_description_t
Processing mass patients in D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks\mass_case_description_t
Processing mass patients in D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks\mass_case_description_t

mass -> Train: 1036, Val: 282, Test: 378


### Step 6 — Merge Classes and Display Shapes

Finally, we concatenate both classes into unified arrays for CNN input.

In [13]:
# Combine classes
X_train = np.concatenate([X_train_calc, X_train_mass], axis=0)
y_train = np.concatenate([y_train_calc, y_train_mass], axis=0)
X_val = np.concatenate([X_val_calc, X_val_mass], axis=0)
y_val = np.concatenate([y_val_calc, y_val_mass], axis=0)
X_test = np.concatenate([X_test_calc, X_test_mass], axis=0)
y_test = np.concatenate([y_test_calc, y_test_mass], axis=0)

print(f"Final shapes -> X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")


Final shapes -> X_train: (2250, 224, 224, 1), X_val: (614, 224, 224, 1), X_test: (704, 224, 224, 1)
